
## CAMADA BRONZE 🥉
- Ingestão de Dados Brutos
- Lendo os dados do catálogo e salvando na nossa camada Bronze


In [0]:
print("Iniciando a ingestão para a Camada Bronze...")

# 1. Leitura direta das tabelas que já estão no seu menu "Catalog" na esquerda
df_orders = spark.table("e_commerce_public.olist_orders_dataset")
df_items = spark.table("e_commerce_public.olist_order_items_dataset")
df_customers = spark.table("e_commerce_public.olist_customers_dataset")
df_products = spark.table("e_commerce_public.olist_products_dataset")

# 2. Salvando como tabelas Delta no banco de dados padrão (default)
df_orders.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze_orders")
df_items.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze_order_items")
df_customers.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze_customers")
df_products.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze_products")

print("Camada Bronze criada com sucesso! 🚀")

# 3. Conferindo rapidamente se a tabela de pedidos foi criada
display(spark.table("bronze_orders").limit(3))

## CAMADA SILVER 🥈
- Limpeza e Cruzamento
- Organização dos Dados
- Retirar Pedidos Cancelados
- União das 4 Tabelas

In [0]:
from pyspark.sql.functions import col, to_timestamp
print("Iniciando do Processo da Camada Silver...")

# 1. Leitura das tabelas Bronze com aliases (apelidos) para manter o código limpo e evitar ambiguidades
df_orders = spark.table("bronze_orders").alias("o")
df_items = spark.table("bronze_order_items").alias("oi")
df_customers = spark.table("bronze_customers").alias("c")
df_products = spark.table("bronze_products").alias("p")

# 2. Padronização: Convertendo a string de data para o formato Timestamp oficial do Spark
df_orders = spark.table("bronze_orders").withColumn(
    "order_purchase_timestamp",
    to_timestamp(col("order_purchase_timestamp"))
).alias("o")

# 3. Regra de Negócio: Filtrando apenas os pedidos que foram efetivamente entregues
order_delivered = df_orders.filter(col("o.order_status") == "delivered")

# 4. Enriquecimento (Joins): Unindo as tabelas
orders_customers = order_delivered.join(
  df_customers, col("o.customer_id") == col("c.customer_id"), "inner"
)

orders_items_enriched = orders_customers.join(
  df_items, col("o.order_id") == col("oi.order_id"), "inner"
)

# Usamos 'left' join aqui para não perder a venda caso o produto tenha sido deletado do catálogo
silver_sales = orders_items_enriched.join(
    df_products, col("oi.product_id") == col("p.product_id"), "left"
)

# 5. Organização: Selecionando estritamente as colunas necessárias para o Dashboard
silver_sales_clean = silver_sales.select(
    col("o.order_id"),
    col("o.customer_id"),
    col("c.customer_unique_id"),
    col("c.customer_state"), # Garantindo que o Estado do cliente está aqui
    col("c.customer_city"),  # Garantindo a Cidade
    col("o.order_purchase_timestamp"),
    col("oi.product_id"),
    col("p.product_category_name"),
    col("oi.price"),
    col("oi.freight_value")
)

# 6. Salvando a tabela consolidada na Camada Silver
silver_sales_clean.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_sales")

print("Camada Silver Processada com Sucesso! 🛠️")

# Conferindo o resultado
display(spark.table("silver_sales").limit(5))

%md
## CAMADA GOLD 🥇
- Limpeza e Cruzamento
- Organização dos Dados
- Retirar Pedidos Cancelados
- União das 4 Tabelas

In [0]:
from pyspark.sql.functions import col, sum, count, date_format, round, desc

print("Iniciando o processamento da Camada GOLD... 🏆")

# 1. Lendo a nossa tabela consolidada da camada Silver
df_silver = spark.table("silver_sales")

# ==========================================
# AGREGAÇÃO 1: Faturamento e Pedidos por Estado
# ==========================================
gold_revenue_by_state = df_silver.groupBy("customer_state") \
  .agg(
    round(sum("price"), 2).alias("total_revenue"),
    count("order_id").alias("total_orders")
  ) \
  .orderBy(desc("total_revenue"))

# Salvando com overwriteSchema por segurança
gold_revenue_by_state.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_revenue_by_state")

print("✅ Tabela 'gold_revenue_by_state' criada!")

# ==========================================
# AGREGAÇÃO 2: Vendas por Mês (Sazonalidade)
# ==========================================
# Extraímos o Ano-Mês (ex: 2017-08) do timestamp da compra
df_silver_month = df_silver.withColumn("year_month", date_format(col("order_purchase_timestamp"), "yyyy-MM"))

gold_sales_by_month = df_silver_month.groupBy("year_month") \
  .agg(
    round(sum("price"), 2).alias("total_revenue"),
    count("order_id").alias("total_orders")
  )\
  .orderBy("year_month")

gold_sales_by_month.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_sales_by_month")
print("✅ Tabela 'gold_sales_by_month' criada!")

# ==========================================
# AGREGAÇÃO 3: Top Categorias de Produtos
# ==========================================
gold_top_categories = df_silver.groupBy("product_category_name") \
  .agg(
    round(sum("price"), 2).alias("total_revenue"),
        count("order_id").alias("total_orders")
  )\
  .orderBy(desc("total_revenue"))

gold_top_categories.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_top_categories")
print("✅ Tabela 'gold_top_categories' criada!")

# Mostrando um preview da tabela de estados para confirmar
display(spark.table("gold_revenue_by_state").limit(10))





In [0]:
# Gráfico 1: Faturamento por Estado
display(spark.table("gold_revenue_by_state"))

In [0]:
# Gráfico 2: Vendas por Mês (Sazonalidade)
display(spark.table("gold_sales_by_month"))

In [0]:
# Gráfico 3: Top Categorias de Produtos
display(spark.table("gold_top_categories"))